# 01_features engineering 

In this notebook, we will make all the feature engineering work to train our models later. 


**Inputs**

Two folder with .txt files, one containing the fan fiction texts and the other the JK.rolling texts. 


**Outputs**

A dataset in csv format containing all the paragraphs from 100 to 300 words in the differents files of our corpus. All the paragraphs will be associated with features, like our target feature (if it's or not a fan_fiction), and stylometrics features that will be useful to train our differents models. 
Another dataset with only 400 lines exctracted from the complete dataset, 200 paragraphs from fanfiction and JK.rolling texts, that will be used to compare the performance between SVM and ChatGPT. 
A tf-idf matrix in pkl format 

## 1. Library import 

In [1]:
# Data Processing
import pandas as pd
import numpy as np

# Feature engineering 
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer

# Other 
from collections import Counter
import os 
import joblib

## 2. Corpus Manipulation 

We will first go through each of the text files in our corpus to select all paragraphs between 100 and 300 words to store them in a dataset. We will associate each paragraph with the variable "category": 0 if it is fanfiction, 1 otherwise.

In [2]:
def dataframe(list_path):

    data = {              # Our futur dataframe
            'text': [], 
            "paragraph" : [], 
            "categorie" : [], 
            "nombre_mots" : []}
    
    cat = -1
    id = 0

    for path in list_path:              # We go through the both folder 
        cat += 1
        
        list_file = os.listdir(path)      
        for file in list_file:              # We go through all the text files in the target folder 
            id += 1

            with open(os.path.join(path, file), encoding= "utf8") as f: # We open the target text file and read it. 
                    content = f.read()
                    paragraphes = content.split('\n')            # We then split the text into differents paragraphs by using "\n"
                    for parag in paragraphes:               # We go through all this paragraphs 
                        mots = parag.split()                # We split those paragraphs to count the words

                        if 100 <= len(mots) and len(mots) <= 300:   # If the paragraph contains between 200 and 300 words, we add it to our data 

                            data["text"].append(f"{file}")
                            data["paragraph"].append(parag)
                            data["categorie"].append(cat)
                            data["nombre_mots"].append(len(mots))

                    print(f"FINISH file n°{id} -- {path} : {file}")

    return pd.DataFrame(data)

In [3]:
# We call the function we made 
df = dataframe(["metrics//corpus//fanfic txt", "metrics//corpus//jk rowling txt"]) #["fanfic txt", "jk rowling txt"]

display(df) 

FINISH file n°1 -- metrics//corpus//fanfic txt : All Our Secrets Laid Bare - firethesound_HP_archive.txt
FINISH file n°2 -- metrics//corpus//fanfic txt : All the Young Dudes - MsKingBean89.txt
FINISH file n°3 -- metrics//corpus//fanfic txt : Amor Vincit Omnia - Twin_Flame_Blues.txt
FINISH file n°4 -- metrics//corpus//fanfic txt : Another Mind Game - May_May_0_0.txt
FINISH file n°5 -- metrics//corpus//fanfic txt : Away Childish Things - lettered.txt
FINISH file n°6 -- metrics//corpus//fanfic txt : Azoth - zeitgeistic.txt
FINISH file n°7 -- metrics//corpus//fanfic txt : BLOODY, SLUTTY, AND PATHETIC - WhatMurdah.txt
FINISH file n°8 -- metrics//corpus//fanfic txt : Breath Mints _ Battle Scars - Onyx_and_Elm.txt
FINISH file n°9 -- metrics//corpus//fanfic txt : Burning Red - NoNameWriter.txt
FINISH file n°10 -- metrics//corpus//fanfic txt : Choices - MesserMoon.txt
FINISH file n°11 -- metrics//corpus//fanfic txt : Draco Malfoy and the Mortifying Ordeal of - isthisselfcare.txt
FINISH file n°1

,text,paragraph,categorie,nombre_mots
0,All Our Secrets Laid Bare - firethesound_HP_ar...,"Frowning at Kingsley’s serious tone, Harry did...",0,101
1,All Our Secrets Laid Bare - firethesound_HP_ar...,But it never took them long to decide that wor...,0,121
2,All Our Secrets Laid Bare - firethesound_HP_ar...,Malfoy had joined up with the Aurors at the sa...,0,128
3,All Our Secrets Laid Bare - firethesound_HP_ar...,"Like everyone else, Harry had assumed that Mal...",0,109
4,All Our Secrets Laid Bare - firethesound_HP_ar...,And that was one hell of an understatement. Wh...,0,102
...,...,...,...,...
11220,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",Harry returned to Gryffindor Tower the followi...,1,220
11221,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...","“Every guest in this Hall,” said Dumbledore, a...",1,103
11222,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",The weather could not have been more different...,1,152
11223,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",When Hermione returned from the trolley and pu...,1,108


In [4]:
print(len(df), len(df[df["categorie"]==0]), len(df[df["categorie"]==1]))

11225 10241 984


## 3. Tokenisation Using Spacy 

Before manipulate the paragraphs to create the features that we will use to train our models, we will use spacy to create a list "docs" that contain all the paragraphs tokenised from our dataset. Those pargraphs tokenised will be usefulls to create the stylometrics features. 

### 3.1 Tokenisation 

In [5]:
nlp = spacy.load("en_core_web_sm")

In [6]:
def process_texts(texts):
    docs = list(nlp.pipe(texts, batch_size=50))
    return docs 

In [7]:
# Docs containing all the pagraphs tokenised 
docs = process_texts(df["paragraph"])


### 3.2 Exctract features 

We now use the "docs" that we previously created to add two columns in our dataset : 

- A column "tokens", containing a list of all the words of the paragraph 

- A columns "pos_tags", containing a list of all the pos_tags (VERB, NOUN...) associated with the words of the target paragraph 

In [8]:
# function to excract the list of token and of pos_tags from an element of "docs"

def extract_features(doc):
    tokens = [token.text for token in doc]
    pos_tags = [token.pos_ for token in doc]

    return tokens, pos_tags 

In [9]:
# function that create our two variables. 
df["tokens"], df["pos_tags"] = zip(
    *[extract_features(doc) for doc in docs]
)


## 4. Features Engineering 

We now have all we need to start the features engineering. We will first create all the stylometrics features (avg_word_length, punctuation_features...), then create the TF-IDF feature. 

### 4.1 Stylometrics features 

#### 4.1.1 Create functions 

We first create a function for each feature that we want to create, and we will call those functions in a second part. 

In [10]:
# Count the number of stop word 
def function_word_freq(tokens):
    if len(tokens) == 0:
        return 0
    
    tokens_lower = [t.lower() for t in tokens]
    count = Counter(tokens_lower)
    total = len(tokens_lower)
    
    return sum(count[word] for word in STOP_WORDS) / total

In [11]:
# 
def type_token_ratio(tokens):
    if len(tokens) == 0:
        return 0
    
    tokens_lower = [t.lower() for t in tokens]
    return len(set(tokens_lower)) / len(tokens_lower)

In [12]:
# Return the average length of the paragraph words 
def avg_word_length(tokens):
    if len(tokens) == 0:
        return 0
    
    return sum(len(t) for t in tokens) / len(tokens)

In [13]:
# Return the number of importants punctuations, like "," "?" and "!"
def punctuation_features(tokens):
    return {
        "exclamation_freq": tokens.count("!") / len(tokens),
        "question_freq": tokens.count("?") / len(tokens),
        "comma_freq": tokens.count(",") / len(tokens),
    }

In [14]:
# Return the average length of the paragraph sentences  
def sentence_length(doc):
    sentences = list(doc.sents)
    return np.mean([len(sent) for sent in sentences])

#### 4.1.2 Call functions 

In [15]:

vectorizer = TfidfVectorizer(
    max_features=200,
    ngram_range=(1,2)
)


In [16]:


# Function to extract stylometric features and create a 
def extract_stylometric_features(doc, tokens, pos_tags):
    total_tokens = len(tokens)
    pos_count = Counter(pos_tags)
    
    # POS ratios
    noun_ratio = pos_count["NOUN"] / total_tokens
    verb_ratio = pos_count["VERB"] / total_tokens
    adj_ratio = pos_count["ADJ"] / total_tokens
    adv_ratio = pos_count["ADV"] / total_tokens

    # Function words
    function_word_score = function_word_freq(tokens)

    # Type-token ratio
    ttr = type_token_ratio(tokens)

    # Avg word length
    avg_len = avg_word_length(tokens)

    # Ponctuation
    punctuation = Counter(tokens)
    exclamation = punctuation["!"] / total_tokens
    question = punctuation["?"] / total_tokens
    comma = punctuation[","] / total_tokens

    # Sentence length
    sentences = list(doc.sents)
    avg_sentence_len = np.mean([len(sent) for sent in sentences]) if sentences else 0

    return [
        function_word_score,
        ttr,
        avg_len,
        noun_ratio,
        verb_ratio,
        adj_ratio,
        adv_ratio,
        exclamation,
        question,
        comma,
        avg_sentence_len
    ]

In [17]:
stylometric_features = []

for doc, tokens, pos_tags in zip(docs, df["tokens"], df["pos_tags"]):
    stylometric_features.append(
        extract_stylometric_features(doc, tokens, pos_tags)
    )

stylometric_features = np.array(stylometric_features)
print(stylometric_features)

[[5.35087719e-01 6.49122807e-01 4.02631579e+00 ... 0.00000000e+00
  3.50877193e-02 2.85000000e+01]
 [5.82089552e-01 6.64179104e-01 4.05223881e+00 ... 0.00000000e+00
  2.98507463e-02 3.35000000e+01]
 [5.91549296e-01 6.33802817e-01 3.98591549e+00 ... 0.00000000e+00
  1.40845070e-02 2.36666667e+01]
 ...
 [4.97142857e-01 6.28571429e-01 4.18285714e+00 ... 0.00000000e+00
  5.71428571e-02 2.18750000e+01]
 [5.84615385e-01 6.92307692e-01 3.76153846e+00 ... 0.00000000e+00
  6.15384615e-02 1.62500000e+01]
 [5.40145985e-01 6.35036496e-01 3.40875912e+00 ... 7.29927007e-03
  2.18978102e-02 1.37000000e+01]]


### 4.2 TF-IDF Feature 

In this section, we will merely compute the TF-IDF of each paragraph 

In [18]:
# Create vectorizer
tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1,2),
    stop_words='english'
)

# Fit + transform
tfidf_matrix = tfidf.fit_transform(df["paragraph"])

### 4.3 Concat data 

We now add all the features we created to our final dataset 

In [19]:

stylometric_columns = [
    "function_word_freq",
    "type_token_ratio",
    "avg_word_length",
    "noun_ratio",
    "verb_ratio",
    "adj_ratio",
    "adv_ratio",
    "exclamation_freq",
    "question_freq",
    "comma_freq",
    "avg_sentence_length"
]
df_stylo = pd.DataFrame(stylometric_features, columns=stylometric_columns)


df = pd.concat([df.reset_index(drop=True), df_stylo], axis=1)

df

,text,paragraph,categorie,nombre_mots,tokens,pos_tags,function_word_freq,type_token_ratio,avg_word_length,noun_ratio,verb_ratio,adj_ratio,adv_ratio,exclamation_freq,question_freq,comma_freq,avg_sentence_length
0,All Our Secrets Laid Bare - firethesound_HP_ar...,"Frowning at Kingsley’s serious tone, Harry did...",0,101,"[Frowning, at, Kingsley, ’s, serious, tone, ,,...","[VERB, ADP, PROPN, PART, ADJ, NOUN, PUNCT, PRO...",0.535088,0.649123,4.026316,0.114035,0.175439,0.061404,0.035088,0.0,0.000000,0.035088,28.500000
1,All Our Secrets Laid Bare - firethesound_HP_ar...,But it never took them long to decide that wor...,0,121,"[But, it, never, took, them, long, to, decide,...","[CCONJ, PRON, ADV, VERB, PRON, ADV, PART, VERB...",0.582090,0.664179,4.052239,0.104478,0.141791,0.044776,0.097015,0.0,0.000000,0.029851,33.500000
2,All Our Secrets Laid Bare - firethesound_HP_ar...,Malfoy had joined up with the Aurors at the sa...,0,128,"[Malfoy, had, joined, up, with, the, Aurors, a...","[PROPN, AUX, VERB, ADP, ADP, DET, PROPN, ADP, ...",0.591549,0.633803,3.985915,0.126761,0.119718,0.056338,0.049296,0.0,0.000000,0.014085,23.666667
3,All Our Secrets Laid Bare - firethesound_HP_ar...,"Like everyone else, Harry had assumed that Mal...",0,109,"[Like, everyone, else, ,, Harry, had, assumed,...","[ADP, PRON, ADV, PUNCT, PROPN, AUX, VERB, SCON...",0.566929,0.669291,4.102362,0.102362,0.133858,0.062992,0.078740,0.0,0.000000,0.062992,25.400000
4,All Our Secrets Laid Bare - firethesound_HP_ar...,And that was one hell of an understatement. Wh...,0,102,"[And, that, was, one, hell, of, an, understate...","[CCONJ, PRON, AUX, NUM, NOUN, ADP, DET, NOUN, ...",0.543103,0.750000,4.474138,0.163793,0.103448,0.068966,0.068966,0.0,0.000000,0.017241,29.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11220,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",Harry returned to Gryffindor Tower the followi...,1,220,"[Harry, returned, to, Gryffindor, Tower, the, ...","[PROPN, VERB, ADP, PROPN, PROPN, DET, VERB, NO...",0.555556,0.584362,4.238683,0.098765,0.156379,0.037037,0.037037,0.0,0.000000,0.032922,22.090909
11221,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...","“Every guest in this Hall,” said Dumbledore, a...",1,103,"[“, Every, guest, in, this, Hall, ,, ”, said, ...","[PUNCT, DET, NOUN, ADP, DET, PROPN, PUNCT, PUN...",0.525000,0.683333,3.800000,0.141667,0.083333,0.058333,0.100000,0.0,0.000000,0.058333,24.000000
11222,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",The weather could not have been more different...,1,152,"[The, weather, could, not, have, been, more, d...","[DET, NOUN, AUX, PART, AUX, AUX, ADV, ADJ, ADP...",0.497143,0.628571,4.182857,0.131429,0.102857,0.040000,0.085714,0.0,0.000000,0.057143,21.875000
11223,"Rowling, J.K - Harry Potter 04 - Harry Pot - R...",When Hermione returned from the trolley and pu...,1,108,"[When, Hermione, returned, from, the, trolley,...","[SCONJ, PROPN, VERB, ADP, DET, NOUN, CCONJ, VE...",0.584615,0.692308,3.761538,0.076923,0.161538,0.030769,0.069231,0.0,0.000000,0.061538,16.250000


## 5. Creation of datasets used to compare model to ChatGPT performances 

### 5.1 data for the train 

We already selected the data to train our model, and we used to evaluate the ChatGPT performances 

In [20]:
# df_down = pd.read_csv("metrics//data//data_fanfic_jk.csv")
# df_down = df_down.sample(frac=1.0)
# # display(df_down)
# df_1 = df_down[df_down["categorie"] == 1][:200]
# df_0 = df_down[df_down["categorie"] == 0][:200]


# df_gpt = pd.concat([df_1, df_0])
# df_gpt = df_gpt.sample(frac=1.0)
# display(df_gpt)


### 5.2 data for the test 

We already compute the data to test the performances of our final model 

In [21]:
# # We then create X_test and y_test by select an equlibrated sample from our global data 

# # Filter the big dataset to not have same data in train and test set 
# # all the line that we have to delete in the big dataset 
# to_sup = df_gpt["paragraph"].tolist()

# # We delete each line 
# for parag in to_sup:
#     df.drop( df[ df['paragraph'] == parag].index, inplace=True)

# # we then shuffle the data 
# df = df.sample(frac=1.0)
# display(df)

# # Select 300 examples from each class
# df_class_0 = df[df["categorie"] == 0].sample(n=300, random_state=42)
# df_class_1 = df[df["categorie"] == 1].sample(n=300, random_state=42)

# # Combine and shuffle
# df_test_gpt = pd.concat([df_class_0, df_class_1]).sample(frac=1, random_state=42).reset_index(drop=True)


## 6. Data Download 

### 6.1 Function creation 

In [22]:
def save_dataframe_as_csv(df: pd.DataFrame, folder_path: str, filename: str = "result.csv"):
    # Download the dataframes as csv files 

    # Create the path to put our result 
    full_path = os.path.join(folder_path, filename)

    # Remove the file if it's already in our folder 
    if os.path.exists(full_path):
        os.remove(full_path)
        print(f"Old file deleted: {full_path}")

    # Download the dataframe as a csv 
    df.to_csv(full_path, index=False)
    print(f"New file saved: {full_path}")

### 6.2 Download ressources 

In [23]:
# Download global data 
save_dataframe_as_csv(df, "metrics//data", "data_fanfic_jk.csv" )

Old file deleted: metrics//data\data_fanfic_jk.csv
New file saved: metrics//data\data_fanfic_jk.csv


In [24]:
# # Download train data for 03_compare_SVM_Chat_GPT
# save_dataframe_as_csv(df_gpt, "metrics//data", "df_gpt.csv" )


In [25]:
# # Download test data for 03_compare_SVM_Chat_GPT
# save_dataframe_as_csv(df_test_gpt, "metrics//data", "df_test_gpt.csv" )

In [26]:
# Save tf-idf vectorizer and matrix 
joblib.dump(tfidf, "metrics/data/tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, "metrics/data/tfidf_matrix.pkl")

['metrics/data/tfidf_matrix.pkl']